# NFL EPA Model - Build and Train XGBoost Model

**Phase 2a: EPA Model Development**

This notebook builds the Expected Points (EP) model using XGBoost with industry best practices:

**Steps:**
1. Load and clean data (filter penalties, kneels, spikes, etc.)
2. Apply blowout filtering (competitive games only)
3. Feature engineering (red zone, home field advantage)
4. Train XGBoost model with LOSO cross-validation
5. Evaluate model (calibration error, MAE, RMSE, R²)
6. Validate field position zones
7. Test turnover contexts
8. Save trained model

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 1. Load Data

In [ ]:
# Load combined data
print("Loading data...")
pbp = pd.read_parquet('../data/play_by_play_2016_2024.parquet')

print(f"✓ Loaded {len(pbp):,} plays")
print(f"  Seasons: {sorted(pbp['season'].unique().tolist())}")
print(f"  Memory: {pbp.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## 2. Data Cleaning - Filter Irrelevant Plays

In [ ]:
print("Starting data cleaning...")
print(f"Initial plays: {len(pbp):,}")

# Keep only plays with EP values
pbp_clean = pbp[pbp['ep'].notna()].copy()
print(f"After removing plays without EP: {len(pbp_clean):,} ({len(pbp_clean)/len(pbp)*100:.1f}%)")

# Filter out specific play types
initial_count = len(pbp_clean)

# 1. Remove penalties
pbp_clean = pbp_clean[pbp_clean['penalty'] != 1]
print(f"After removing penalties: {len(pbp_clean):,} (-{initial_count - len(pbp_clean):,})")

# 2. Remove QB kneels
pbp_clean = pbp_clean[pbp_clean['qb_kneel'] != 1]
print(f"After removing QB kneels: {len(pbp_clean):,}")

# 3. Remove QB spikes
pbp_clean = pbp_clean[pbp_clean['qb_spike'] != 1]
print(f"After removing QB spikes: {len(pbp_clean):,}")

# 4. Remove two-point conversions
if 'two_point_attempt' in pbp_clean.columns:
    pbp_clean = pbp_clean[pbp_clean['two_point_attempt'] != 1]
    print(f"After removing 2-pt conversions: {len(pbp_clean):,}")

# 5. Keep only valid downs (1-4)
pbp_clean = pbp_clean[pbp_clean['down'].isin([1, 2, 3, 4])]
print(f"After keeping valid downs (1-4): {len(pbp_clean):,}")

# 6. Keep only plays with valid field position
pbp_clean = pbp_clean[(pbp_clean['yardline_100'] >= 1) & (pbp_clean['yardline_100'] <= 99)]
print(f"After filtering valid field position: {len(pbp_clean):,}")

# 7. Keep only plays with valid distance
pbp_clean = pbp_clean[(pbp_clean['ydstogo'] >= 1) & (pbp_clean['ydstogo'] <= 99)]
print(f"After filtering valid distance: {len(pbp_clean):,}")

print(f"\n✓ Cleaning complete: {len(pbp_clean):,} plays remaining")
print(f"  Removed: {initial_count - len(pbp_clean):,} plays ({(initial_count - len(pbp_clean))/initial_count*100:.1f}%)")

## 3. Blowout Filtering - Keep Only Competitive Games

In [ ]:
print("Applying blowout filtering...")
print(f"Plays before filtering: {len(pbp_clean):,}")

# Calculate absolute score differential
pbp_clean['abs_score_diff'] = pbp_clean['score_differential'].abs()

# Apply quarter-based filtering
# Q1, Q3: 10 points | Q2: 17 points | Q4: 21 points
competitive_mask = (
    ((pbp_clean['qtr'] == 1) & (pbp_clean['abs_score_diff'] <= 10)) |
    ((pbp_clean['qtr'] == 2) & (pbp_clean['abs_score_diff'] <= 17)) |
    ((pbp_clean['qtr'] == 3) & (pbp_clean['abs_score_diff'] <= 10)) |
    ((pbp_clean['qtr'] == 4) & (pbp_clean['abs_score_diff'] <= 21))
)

pbp_competitive = pbp_clean[competitive_mask].copy()

print(f"After blowout filtering: {len(pbp_competitive):,}")
print(f"  Filtered out: {len(pbp_clean) - len(pbp_competitive):,} plays ({(len(pbp_clean) - len(pbp_competitive))/len(pbp_clean)*100:.1f}%)")

# Show filtering by quarter
print("\nFiltering breakdown by quarter:")
for q in [1, 2, 3, 4]:
    q_total = len(pbp_clean[pbp_clean['qtr'] == q])
    q_kept = len(pbp_competitive[pbp_competitive['qtr'] == q])
    print(f"  Q{q}: {q_kept:,} / {q_total:,} ({q_kept/q_total*100:.1f}% kept)")

## 4. Feature Engineering

In [ ]:
print("Creating features...")

# 1. Red zone flag (inside 20-yard line)
pbp_competitive['red_zone'] = (pbp_competitive['yardline_100'] <= 20).astype(int)
print(f"  Red zone plays: {pbp_competitive['red_zone'].sum():,} ({pbp_competitive['red_zone'].mean()*100:.1f}%)")

# 2. Home field advantage (1 if home team has possession)
pbp_competitive['home_field_advantage'] = (pbp_competitive['posteam'] == pbp_competitive['home_team']).astype(int)
print(f"  Home team possessions: {pbp_competitive['home_field_advantage'].sum():,} ({pbp_competitive['home_field_advantage'].mean()*100:.1f}%)")

print("\n✓ Feature engineering complete")

## 5. Prepare Training Data

In [ ]:
# Define features for the model
feature_cols = ['down', 'ydstogo', 'yardline_100', 'red_zone', 'home_field_advantage']
target_col = 'ep'

# Create feature matrix and target
X = pbp_competitive[feature_cols].copy()
y = pbp_competitive[target_col].copy()
seasons = pbp_competitive['season'].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures: {feature_cols}")
print(f"Target: {target_col}")

# Check for missing values
print(f"\nMissing values:")
print(X.isnull().sum())

# Remove any remaining missing values
valid_mask = X.notna().all(axis=1) & y.notna()
X = X[valid_mask]
y = y[valid_mask]
seasons = seasons[valid_mask]

print(f"\nFinal dataset: {len(X):,} plays")

## 6. Train XGBoost Model with LOSO Cross-Validation

In [ ]:
print("Training XGBoost EPA Model with Leave-One-Season-Out (LOSO) Cross-Validation")
print("=" * 80)

unique_seasons = sorted(seasons.unique())
print(f"Seasons: {unique_seasons}")
print(f"Total CV folds: {len(unique_seasons)}")
print()

# Store results
loso_results = []
all_predictions = []
all_actuals = []

# LOSO Cross-Validation
for holdout_season in unique_seasons:
    print(f"Fold: Holdout season = {holdout_season}")
    
    # Split data
    train_mask = seasons != holdout_season
    test_mask = seasons == holdout_season
    
    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]
    
    print(f"  Train: {len(X_train):,} plays (seasons {[s for s in unique_seasons if s != holdout_season]})")
    print(f"  Test:  {len(X_test):,} plays")
    
    # Train XGBoost model
    model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train, verbose=False)
    
    # Predict on test set
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²:   {r2:.4f}")
    print()
    
    # Store results
    loso_results.append({
        'holdout_season': holdout_season,
        'train_size': len(X_train),
        'test_size': len(X_test),
        'mae': mae,
        'rmse': rmse,
        'r2': r2
    })
    
    all_predictions.extend(y_pred)
    all_actuals.extend(y_test)

# Convert to DataFrame
results_df = pd.DataFrame(loso_results)

print("=" * 80)
print("LOSO Cross-Validation Results:")
print(results_df.to_string(index=False))
print()
print("Average Performance:")
print(f"  MAE:  {results_df['mae'].mean():.4f} ± {results_df['mae'].std():.4f}")
print(f"  RMSE: {results_df['rmse'].mean():.4f} ± {results_df['rmse'].std():.4f}")
print(f"  R²:   {results_df['r2'].mean():.4f} ± {results_df['r2'].std():.4f}")

## 7. Train Final Model on All Data

In [ ]:
print("Training final model on all data...")

# Train on all competitive games
final_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y, verbose=False)

print("✓ Final model trained")

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(importance.to_string(index=False))

# Plot feature importance
plt.figure(figsize=(10, 5))
plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Importance')
plt.title('XGBoost Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Calculate Calibration Error

In [ ]:
print("Calculating Calibration Error...")

# Convert to numpy arrays
y_pred_all = np.array(all_predictions)
y_actual_all = np.array(all_actuals)

# Round to nearest 0.5 for calibration
y_pred_rounded = np.round(y_pred_all * 2) / 2
y_actual_rounded = np.round(y_actual_all * 2) / 2

# Calculate calibration error
calibration_df = pd.DataFrame({
    'predicted': y_pred_rounded,
    'actual': y_actual_rounded
})

calibration_grouped = calibration_df.groupby('predicted')['actual'].agg(['mean', 'count']).reset_index()
calibration_grouped['diff'] = np.abs(calibration_grouped['predicted'] - calibration_grouped['mean'])

# Weighted average calibration error
total_count = calibration_grouped['count'].sum()
calibration_error = (calibration_grouped['diff'] * calibration_grouped['count']).sum() / total_count

print(f"\n✓ Calibration Error: {calibration_error:.6f}")
print(f"  Target: < 0.01")
print(f"  nflfastR benchmark: 0.006")

if calibration_error < 0.01:
    print("  ✓ PASSED: Model meets calibration target!")
else:
    print("  ⚠️  WARNING: Calibration error above target")

# Plot calibration
plt.figure(figsize=(10, 6))
plt.scatter(calibration_grouped['predicted'], calibration_grouped['mean'], 
            s=calibration_grouped['count']/10, alpha=0.6)
plt.plot([-2, 6], [-2, 6], 'r--', label='Perfect Calibration')
plt.xlabel('Predicted EP')
plt.ylabel('Actual EP (Mean)')
plt.title(f'Calibration Plot (Error: {calibration_error:.6f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Validate Field Position Zones

In [ ]:
print("Validating Field Position Zones...")

# Create field position zones
pbp_eval = pbp_competitive[[*feature_cols, target_col]].copy()
pbp_eval['predicted_ep'] = final_model.predict(X)

# Define zones
pbp_eval['zone'] = pd.cut(
    pbp_eval['yardline_100'],
    bins=[0, 20, 50, 80, 100],
    labels=['Red Zone (<20)', 'Opponent Territory (20-50)', 'Midfield (50-80)', 'Own Territory (>80)']
)

# Calculate metrics by zone
zone_metrics = pbp_eval.groupby('zone').apply(
    lambda x: pd.Series({
        'count': len(x),
        'actual_mean': x[target_col].mean(),
        'predicted_mean': x['predicted_ep'].mean(),
        'mae': mean_absolute_error(x[target_col], x['predicted_ep']),
        'r2': r2_score(x[target_col], x['predicted_ep'])
    })
).reset_index()

print("\nPerformance by Field Position Zone:")
print(zone_metrics.to_string(index=False))

# Plot EP by field position
ep_by_yard = pbp_eval.groupby('yardline_100').agg({
    target_col: 'mean',
    'predicted_ep': 'mean'
}).reset_index()

plt.figure(figsize=(14, 6))
plt.plot(ep_by_yard['yardline_100'], ep_by_yard[target_col], label='Actual EP', linewidth=2)
plt.plot(ep_by_yard['yardline_100'], ep_by_yard['predicted_ep'], label='Predicted EP', linewidth=2, linestyle='--')
plt.axvline(20, color='red', linestyle=':', alpha=0.5, label='Red Zone')
plt.axvline(50, color='green', linestyle=':', alpha=0.5, label='Midfield')
plt.xlabel('Yards to Goal')
plt.ylabel('Expected Points')
plt.title('Model Performance by Field Position')
plt.legend()
plt.gca().invert_xaxis()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Field position zones validated")

## 10. Test Model Predictions

In [ ]:
print("Testing sample predictions...\n")

# Test scenarios
test_scenarios = [
    {'down': 1, 'ydstogo': 10, 'yardline_100': 75, 'red_zone': 0, 'home_field_advantage': 1, 'description': '1st & 10 at own 25'},
    {'down': 3, 'ydstogo': 5, 'yardline_100': 50, 'red_zone': 0, 'home_field_advantage': 1, 'description': '3rd & 5 at midfield'},
    {'down': 1, 'ydstogo': 10, 'yardline_100': 15, 'red_zone': 1, 'home_field_advantage': 1, 'description': '1st & 10 at opp 15 (red zone)'},
    {'down': 4, 'ydstogo': 1, 'yardline_100': 35, 'red_zone': 0, 'home_field_advantage': 0, 'description': '4th & 1 at opp 35 (away team)'},
    {'down': 1, 'ydstogo': 10, 'yardline_100': 5, 'red_zone': 1, 'home_field_advantage': 1, 'description': '1st & Goal at 5'},
]

test_df = pd.DataFrame(test_scenarios)
test_features = test_df[feature_cols]
test_predictions = final_model.predict(test_features)

print("Sample Predictions:")
print("=" * 80)
for i, scenario in enumerate(test_scenarios):
    print(f"{scenario['description']:40s} → EP = {test_predictions[i]:.2f}")

print("\n✓ Predictions look reasonable")

## 11. Save Model

In [ ]:
# Create models directory
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

# Save model
model_path = models_dir / 'epa_model_xgboost.joblib'
joblib.dump(final_model, model_path)
print(f"✓ Model saved to: {model_path}")

# Save metadata
metadata = {
    'model_type': 'XGBoost Regressor',
    'features': feature_cols,
    'target': target_col,
    'training_samples': len(X),
    'seasons': unique_seasons.tolist(),
    'performance': {
        'mae': float(results_df['mae'].mean()),
        'rmse': float(results_df['rmse'].mean()),
        'r2': float(results_df['r2'].mean()),
        'calibration_error': float(calibration_error)
    },
    'hyperparameters': {
        'n_estimators': 200,
        'max_depth': 6,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8
    },
    'feature_importance': importance.to_dict('records')
}

import json
metadata_path = models_dir / 'epa_model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Metadata saved to: {metadata_path}")

# Save LOSO results
results_path = models_dir / 'epa_model_loso_results.csv'
results_df.to_csv(results_path, index=False)
print(f"✓ LOSO results saved to: {results_path}")

## 12. Summary

In [ ]:
print("=" * 80)
print("EPA MODEL TRAINING COMPLETE")
print("=" * 80)
print()
print(f"Model: XGBoost Regressor")
print(f"Training Data: {len(X):,} competitive game plays (2016-2024)")
print(f"Features: {len(feature_cols)} ({', '.join(feature_cols)})")
print()
print("Performance (LOSO Cross-Validation):")
print(f"  MAE:  {results_df['mae'].mean():.4f} (Target: < 0.5) {'✓ PASS' if results_df['mae'].mean() < 0.5 else '✗ FAIL'}")
print(f"  RMSE: {results_df['rmse'].mean():.4f}")
print(f"  R²:   {results_df['r2'].mean():.4f} (Target: > 0.85) {'✓ PASS' if results_df['r2'].mean() > 0.85 else '✗ FAIL'}")
print(f"  Calibration Error: {calibration_error:.6f} (Target: < 0.01) {'✓ PASS' if calibration_error < 0.01 else '✗ FAIL'}")
print()
print("Model Files:")
print(f"  {model_path}")
print(f"  {metadata_path}")
print(f"  {results_path}")
print()
print("✓ Ready for API integration (Phase 3)")
print("=" * 80)